# Cluster reads by accessibility features around CAGE TSS, per footprint class

Single sample (AL10), no PI/gene filtering. TSS anchors are FANTOM5 LCL consensus CAGE peaks.
For every (read, TSS) pair where the read carries at least one footprint within +/- HALF_WINDOW bp
of the TSS, build a per-class feature vector:

- Footprint geometry per class (PPP / PIC / nucleosome_140-160 / FIRE_nucleosome / unknown):
  count, upstream/downstream count, total span_bp, nearest mid-distance, nearest edge-distance.
  Nucleosome classes additionally get +1 / -1 distance & span.
- m6A signal: window-level count of modified m6A positions plus per-class count of modified
  m6A positions falling inside footprints of that class. m6A comes from the
  ft_extracted_m6a BED12s, which contain only modified positions, so this is a
  modified-only proxy for accessibility (no unmodified denominator).

Then PCA -> k-means with elbow / silhouette selection.

### 1. Setup, TSS list. Load FANTOM5 LCL consensus CAGE peaks as the TSS universe; locate footprint and m6A BEDs for the sample.

In [4]:
import os
import re
import gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import pysam
import matplotlib.pyplot as plt

SAMPLE = "AL10_bc2178_19130"
HALF_WINDOW = 500   # change to 250 for a narrower window

CAGE_TSS_BED = Path(
    "/project/spott/cshan/annotations/fantom5/"
    "fantom5.hg38.LCL.consensus.CAGE_peaks.bed.gz"
)
FP_BED_ROOT = Path(
    "/project/spott/cshan/fiber-seq/results/PolII/footprint_summary_beds"
)
M6A_BED_DIR = Path(
    f"/project/spott/1_Shared_projects/LCL_Fiber_seq/FIRE/results/{SAMPLE}/"
    "extracted_results/m6a_by_chr"
)

FP_CLASSES = ["PPP", "PIC", "nucleosome_140-160", "FIRE_nucleosome", "unknown"]
NUCLEOSOME_CLASSES = {"FIRE_nucleosome", "nucleosome_140-160"}

OUT_ROOT = Path(
    f"/project/spott/cshan/fiber-seq/results/PolII/cluster_reads_by_FP_features/"
    f"{SAMPLE}_w{HALF_WINDOW}"
)
PER_CHROM_DIR = OUT_ROOT / "per_chrom_features"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
PER_CHROM_DIR.mkdir(parents=True, exist_ok=True)

# Load FANTOM5 CAGE peaks (BED9). Restrict to chromosomes that have footprint BEDs.
cage = pd.read_csv(
    CAGE_TSS_BED, sep="\t", header=None,
    usecols=[0, 1, 2, 3, 4, 5],
    names=["chrom", "start", "end", "peak_id", "score", "strand"],
    dtype={"chrom": str, "start": np.int64, "end": np.int64,
           "peak_id": str, "score": str, "strand": str},
)
fp_chroms = {p.name.replace("_footprints.bed.gz", "")
             for p in (FP_BED_ROOT / FP_CLASSES[0]).glob("*_footprints.bed.gz")}
cage = cage[cage["chrom"].isin(fp_chroms)].copy()
cage["tss_coordinate"] = cage["start"].astype(np.int64)
cage = cage[["chrom", "tss_coordinate", "strand", "peak_id"]].reset_index(drop=True)
print(f"CAGE peaks on FP-supported chromosomes: {len(cage):,} "
      f"across {cage['chrom'].nunique()} chroms")
print(cage.head(3).to_string())

CAGE peaks on FP-supported chromosomes: 108,067 across 23 chroms
  chrom  tss_coordinate strand                peak_id
0  chr1          634008      +  chr1:634008..634009,+
1  chr1          634009      +  chr1:634009..634010,+
2  chr1          634026      +  chr1:634026..634027,+


### 2. Extract per-(read, TSS) features per chromosome.

Heavy step (per-class tabix sweeps over all CAGE TSS) — runs as a SLURM job
rather than in the notebook kernel. Logic lives in:

- `cluster_reads_code/cluster_reads_by_FP_features_extract.py` — for each TSS,
  queries the 5 tabix-indexed footprint BEDs and the m6A BED12 within
  +/- HALF_WINDOW and aggregates per read x class. Writes one parquet per
  chromosome under `{OUT_ROOT}/per_chrom_features/`. Skips chromosomes whose
  parquet already exists.
- `cluster_reads_code/cluster_reads_by_FP_features_extract.sh` — SLURM wrapper
  (caslake, 180G, 24h).

Submit:

```
sbatch /project/spott/cshan/fiber-seq/code/PolII_footprints_code/cluster_reads_code/cluster_reads_by_FP_features_extract.sh
```

The cell below just shows the script paths and lists per-chromosome parquets
the job has produced so far.

In [ ]:
EXTRACT_PY = Path(
    "/project/spott/cshan/fiber-seq/code/PolII_footprints_code/"
    "cluster_reads_code/cluster_reads_by_FP_features_extract.py"
)
EXTRACT_SH = Path(
    "/project/spott/cshan/fiber-seq/code/PolII_footprints_code/"
    "cluster_reads_code/cluster_reads_by_FP_features_extract.sh"
)
print(f"Extraction script: {EXTRACT_PY}")
print(f"SLURM wrapper:     {EXTRACT_SH}")
print(f"To submit:  sbatch {EXTRACT_SH}\n")

done = sorted(PER_CHROM_DIR.glob("*_features.parquet"))
print(f"Per-chrom parquets present in {PER_CHROM_DIR}: {len(done)}")
for p in done:
    print(f"  {p.name}")
missing = sorted(set(cage["chrom"].unique()) - {p.name.replace("_features.parquet", "") for p in done})
if missing:
    print(f"\nMissing ({len(missing)}): {missing}")
else:
    print("\nAll chromosomes have feature parquets.")

### 3. Concatenate per-chromosome features into one (read, TSS) table.

In [ ]:
feature_paths = sorted(PER_CHROM_DIR.glob("*_features.parquet"))
if not feature_paths:
    raise FileNotFoundError(f"No feature parquets in {PER_CHROM_DIR}")

parts = []
for p in feature_paths:
    df = pd.read_parquet(p)
    if not df.empty:
        parts.append(df)
    print(f"  {p.name}: {len(df):,} rows")
joined = pd.concat(parts, ignore_index=True)
del parts
print(f"\nTotal (read, TSS) feature rows: {len(joined):,}")
print(f"Unique reads: {joined['read_core'].nunique():,}")
print(f"Unique TSS:   {joined['peak_id'].nunique():,}")
joined.to_parquet(OUT_ROOT / "read_features.parquet", index=False)
print(joined.head(2).T.to_string())

### 4. Build the feature matrix. log1p count-like features; sentinel-impute and z-score continuous distance/span features.

In [ ]:
joined = pd.read_parquet(OUT_ROOT / "read_features.parquet")

count_cols, dist_cols, span_cols = [], [], []
for cls in FP_CLASSES:
    count_cols.extend([f"{cls}_count", f"{cls}_upstream_count",
                       f"{cls}_downstream_count", f"{cls}_span_bp",
                       f"{cls}_m6a_mod_in_class"])
    dist_cols.extend([f"{cls}_nearest_mid_dist", f"{cls}_nearest_edge_dist"])
    if cls in NUCLEOSOME_CLASSES:
        dist_cols.extend([f"{cls}_plus1_dist", f"{cls}_minus1_dist"])
        span_cols.extend([f"{cls}_plus1_span", f"{cls}_minus1_span"])
count_cols.append("n_m6a_mod_window")

print(f"{len(count_cols)} count-like, {len(dist_cols)} distance, "
      f"{len(span_cols)} span features")

# log1p for count-like features
X_counts = np.log1p(joined[count_cols].to_numpy(dtype=np.float32))

# Distance imputation: missing nearest_*_dist -> HALF_WINDOW (no nearby footprint).
# Missing minus1_dist (signed -ve) -> -HALF_WINDOW. Plus1_dist already +ve.
dist_block = joined[dist_cols].to_numpy(dtype=np.float32, copy=True)
for i, c in enumerate(dist_cols):
    col = dist_block[:, i]
    nan_mask = np.isnan(col)
    if c.endswith("_minus1_dist"):
        col[nan_mask] = -HALF_WINDOW
    else:
        col[nan_mask] = HALF_WINDOW
    dist_block[:, i] = col

span_block = joined[span_cols].to_numpy(dtype=np.float32, copy=True)
span_block[np.isnan(span_block)] = 0.0

cont_block = np.hstack([dist_block, span_block])
cont_cols = dist_cols + span_cols
mu = cont_block.mean(axis=0)
sd = cont_block.std(axis=0)
sd[sd == 0] = 1.0
X_cont = (cont_block - mu) / sd

X = np.hstack([X_counts, X_cont]).astype(np.float32)
feature_names = count_cols + cont_cols
print(f"Feature matrix: {X.shape}")

np.save(OUT_ROOT / "X.npy", X)
with open(OUT_ROOT / "feature_names.txt", "w") as f:
    f.write("\n".join(feature_names) + "\n")

meta_keep = ["read_core", "peak_id", "chromosome", "tss_coordinate", "tss_strand"]
meta_aligned = joined[meta_keep].copy()
meta_aligned.to_parquet(OUT_ROOT / "X_metadata.parquet", index=False)
print("Wrote X.npy, feature_names.txt, X_metadata.parquet")

### 5. PCA for a compact clustering space, plus a 2D embedding for visualization.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

X = np.load(OUT_ROOT / "X.npy", mmap_mode="r")
meta_aligned = pd.read_parquet(OUT_ROOT / "X_metadata.parquet")
print(f"X shape: {X.shape}")

N_PCA_COMPONENTS = min(20, X.shape[1])
PCA_FIT_MAX = 500_000
rng = np.random.default_rng(0)

if X.shape[0] > PCA_FIT_MAX:
    fit_idx = rng.choice(X.shape[0], size=PCA_FIT_MAX, replace=False)
    fit_idx.sort()
    pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0).fit(X[fit_idx])
else:
    pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0).fit(X)

chunk = 200_000
X_pca = np.empty((X.shape[0], N_PCA_COMPONENTS), dtype=np.float32)
for i in range(0, X.shape[0], chunk):
    X_pca[i:i + chunk] = pca.transform(X[i:i + chunk]).astype(np.float32)

explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)
print(f"Cumulative explained variance: {cum_explained.round(3)}")
for threshold in (0.80, 0.90, 0.95):
    if cum_explained[-1] >= threshold:
        n_pc = int(np.searchsorted(cum_explained, threshold) + 1)
        print(f"{int(threshold*100)}% variance reached at PC{n_pc}")
    else:
        print(f"{int(threshold*100)}% variance not reached within {N_PCA_COMPONENTS} PCs")
np.save(OUT_ROOT / "X_pca.npy", X_pca)

pcs = np.arange(1, N_PCA_COMPONENTS + 1)
fig, ax1 = plt.subplots(figsize=(7, 3.5))
ax1.bar(pcs, explained * 100, color="#4C78A8", alpha=0.75, label="Per PC")
ax1.set_xlabel("PC"); ax1.set_ylabel("Explained variance per PC (%)")
ax1.set_xticks(pcs)
ax2 = ax1.twinx()
ax2.plot(pcs, cum_explained * 100, color="#E15759", marker="o", label="Cumulative")
ax2.set_ylabel("Cumulative explained variance (%)")
for threshold in (80, 90, 95):
    ax2.axhline(threshold, color="0.65", linestyle="--", linewidth=0.8)
    ax2.text(pcs[-1] + 0.15, threshold, f"{threshold}%", va="center", fontsize=8)
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center right", frameon=False)
ax1.set_title("PCA explained variance")
plt.tight_layout(); plt.show()

try:
    plot_choice = input("Choose plot [pca/umap/tsne/none] (default pca): ").strip().lower()
except EOFError:
    plot_choice = ""
plot_choice = plot_choice or "pca"
if plot_choice not in {"pca", "umap", "tsne", "none"}:
    print(f"Unknown plot choice {plot_choice!r}; using 'pca'."); plot_choice = "pca"
print(f"Plot choice: {plot_choice}")

PLOT_N = min(20_000, X_pca.shape[0])
viz_idx = rng.choice(X_pca.shape[0], size=PLOT_N, replace=False); viz_idx.sort()
X_viz = X_pca[viz_idx]

def _plot_2d(emb, title, xlab, ylab):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(emb[:, 0], emb[:, 1], s=4, alpha=0.4, color="#4C78A8")
    ax.set_title(title); ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    plt.tight_layout(); plt.show()

if plot_choice == "pca":
    _plot_2d(X_viz[:, :2], "PCA", "PC1", "PC2")
elif plot_choice == "umap":
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="IProgress not found.*")
        warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
        import umap
        reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0,
                            n_components=2, low_memory=True)
        umap_emb = reducer.fit_transform(X_viz)
    np.save(OUT_ROOT / "umap_embedding.npy", umap_emb)
    np.save(OUT_ROOT / "umap_indices.npy", viz_idx)
    _plot_2d(umap_emb, "UMAP", "UMAP1", "UMAP2")
elif plot_choice == "tsne":
    tsne_emb = TSNE(n_components=2, perplexity=30, init="pca",
                    learning_rate="auto", random_state=0).fit_transform(X_viz)
    np.save(OUT_ROOT / "tsne_embedding.npy", tsne_emb)
    np.save(OUT_ROOT / "tsne_indices.npy", viz_idx)
    _plot_2d(tsne_emb, "t-SNE", "t-SNE1", "t-SNE2")
else:
    print("Skipping 2D embedding plot.")

### 6. K-means across a range of k. Inertia (elbow) on the full PCA matrix; silhouette on a 20k subsample. Final clustering uses the chosen k re-fit on the full matrix.

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(0)
X_pca = np.load(OUT_ROOT / "X_pca.npy")
meta_aligned = pd.read_parquet(OUT_ROOT / "X_metadata.parquet")
print(f"X_pca shape: {X_pca.shape}")

K_RANGE = list(range(2, 11))
SIL_N = min(20_000, X_pca.shape[0])
sil_idx = rng.choice(X_pca.shape[0], size=SIL_N, replace=False); sil_idx.sort()
X_sil = X_pca[sil_idx]

inertias, silhouettes, labels_by_k = [], [], {}
for k in K_RANGE:
    km = MiniBatchKMeans(n_clusters=k, batch_size=10_000, random_state=0,
                        n_init=5, max_iter=200)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_sil, labels[sil_idx])
    silhouettes.append(sil)
    labels_by_k[k] = labels
    print(f"k={k:2d}  inertia={km.inertia_:,.0f}  silhouette={sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(K_RANGE, inertias, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow")
axes[1].plot(K_RANGE, silhouettes, marker="o", color="#E15759")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette (subsample)")
axes[1].set_title("Silhouette")
plt.tight_layout(); plt.show()

best_k = int(K_RANGE[int(np.argmax(silhouettes))])
print(f"Best k by silhouette: {best_k}")
final_labels = labels_by_k[best_k]
np.save(OUT_ROOT / f"kmeans_labels_k{best_k}.npy", final_labels)

available_plots = ["pca"]
for name in ("umap", "tsne"):
    if (OUT_ROOT / f"{name}_embedding.npy").exists() and (OUT_ROOT / f"{name}_indices.npy").exists():
        available_plots.append(name)
try:
    cluster_plot = input(
        f"Choose cluster plot {available_plots + ['none']} (default pca): "
    ).strip().lower()
except EOFError:
    cluster_plot = ""
cluster_plot = cluster_plot or "pca"
if cluster_plot not in set(available_plots + ["none"]):
    print(f"Unknown choice {cluster_plot!r}; using 'pca'."); cluster_plot = "pca"
print(f"Cluster plot choice: {cluster_plot}")

if cluster_plot == "pca":
    plot_n = min(20_000, X_pca.shape[0])
    plot_idx = rng.choice(X_pca.shape[0], size=plot_n, replace=False); plot_idx.sort()
    plot_emb = X_pca[plot_idx, :2]; xlab, ylab = "PC1", "PC2"
    plot_title = f"PC1/PC2 colored by k-means (k={best_k})"
elif cluster_plot in {"umap", "tsne"}:
    plot_emb = np.load(OUT_ROOT / f"{cluster_plot}_embedding.npy")
    plot_idx = np.load(OUT_ROOT / f"{cluster_plot}_indices.npy")
    plot_title = f"{cluster_plot.upper()} colored by k-means (k={best_k})"
    xlab, ylab = f"{cluster_plot.upper()}1", f"{cluster_plot.upper()}2"
else:
    plot_emb = None

if plot_emb is not None:
    plot_labels = final_labels[plot_idx]
    fig, ax = plt.subplots(figsize=(6, 5))
    cmap = plt.cm.get_cmap("tab10", best_k)
    for c in range(best_k):
        m = plot_labels == c
        ax.scatter(plot_emb[m, 0], plot_emb[m, 1], s=4, alpha=0.5,
                   color=cmap(c), label=f"c{c}")
    ax.set_title(plot_title); ax.set_xlabel(xlab); ax.set_ylabel(ylab)
    ax.legend(markerscale=3, fontsize=8, frameon=False, loc="best")
    plt.tight_layout(); plt.show()
else:
    print("Skipping cluster embedding plot.")

meta_aligned["cluster"] = final_labels
print("\nCluster sizes:")
print(meta_aligned["cluster"].value_counts().sort_index().to_string())

feature_names = open(OUT_ROOT / "feature_names.txt").read().splitlines()
X = np.load(OUT_ROOT / "X.npy", mmap_mode="r")
cluster_means = pd.DataFrame(
    {f"c{c}": X[final_labels == c].mean(axis=0) for c in range(best_k)},
    index=feature_names,
)
print("\nCluster centroid means (standardized feature space):")
print(cluster_means.round(2).to_string())
cluster_means.to_csv(OUT_ROOT / f"cluster_centroid_means_k{best_k}.tsv", sep="\t")
meta_aligned.to_parquet(OUT_ROOT / f"X_metadata_with_cluster_k{best_k}.parquet", index=False)